In [67]:
# --- STEP 0: System Setup ---
! git clone https://github.com/sreedharkaranam-ai/edu-101-java-code.git
print("🔧 Installing Java 17, Temporal CLI, and Maven...")
!sudo apt-get update -y > /dev/null
!sudo apt-get install openjdk-17-jdk-headless maven -y > /dev/null
!ls

fatal: destination path 'edu-101-java-code' already exists and is not an empty directory.
🔧 Installing Java 17, Temporal CLI, and Maven...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
build	      docker		 gradlew      settings.gradle
build.gradle  edu-101-java-code  gradlew.bat  springboot
cluster2.db   gradle		 LICENSE      springboot-basic
core	      gradle.properties  README.md


In [68]:
# 1. Download using the correct 2026 URL
!curl -sSf https://temporal.download/cli.sh | sh

# 2. Add the binary to the Python environment's PATH
import os
temporal_bin_path = os.path.expanduser("~/.temporalio/bin")
os.environ['PATH'] += f":{temporal_bin_path}"

# 3. Create a system-wide link so all '!temporal' commands work seamlessly
!ln -sf /root/.temporalio/bin/temporal /usr/local/bin/temporal

# 4. Verify it works!
print("\n✅ Verification:")
!temporal --version

%cd /content/edu-101-java-code/example1

temporal: Downloading Temporal CLI latest
temporal: Temporal CLI installed at /root/.temporalio/bin/temporal
temporal: Start the server with: temporal server start-dev
temporal: Or start a workflow with: temporal workflow start
temporal: For usage, run: temporal --help

✅ Verification:
temporal version 1.6.1 (Server 1.30.1, UI 2.45.3)
/content/edu-101-java-code/example1


## --- EXAMPLE 1 ---

In [69]:
import subprocess
import time

# Cleanup
!pkill -9 temporal || true
!pkill -f java || true

print("\n🏗️ Setting up Example 1...")
# Make sure we are in the right folder!
%cd /content/edu-101-java-code/example1

print("🛰️ Terminal 1: Starting Temporal Server (Port 8081)...")
# Popen starts the server in the background and moves to the next line immediately
server_process = subprocess.Popen(['temporal', 'server', 'start-dev', '--ui-port', '8081'])

print("👷 Terminal 2: Compiling...")
!mvn clean compile > /dev/null

print("👷 Terminal 2: Starting Java Worker...")
# Run the worker in the background too
worker_process = subprocess.Popen(['mvn', 'exec:java', '-Dexec.mainClass=helloworkflow.SayHelloWorker'])

print("⏳ Waiting 15 seconds for server and worker to start...")
time.sleep(15)

print("🎬 Terminal 3: Running Starter...")
# We use ! for the starter because we DO want the notebook to wait for this one to finish
!mvn exec:java -Dexec.mainClass="helloworkflow.Starter"

^C

🏗️ Setting up Example 1...
/content/edu-101-java-code/example1
🛰️ Terminal 1: Starting Temporal Server (Port 8081)...
👷 Terminal 2: Compiling...
👷 Terminal 2: Starting Java Worker...
⏳ Waiting 15 seconds for server and worker to start...
🎬 Terminal 3: Running Starter...
[INFO] Scanning for projects...
[INFO] 
[INFO] ---------------------< com.example:temporal-demo >----------------------
[INFO] Building temporal-demo 1.0-SNAPSHOT
[INFO] --------------------------------[ jar ]---------------------------------
[INFO] 
[INFO] --- exec-maven-plugin:3.6.3:java (default-cli) @ temporal-demo ---
[helloworkflow.Starter.main()] INFO io.temporal.serviceclient.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}}
Workflow result: Hello Temporal
[WARNING] thread Thread[grpc-connection-manager-thread-0,5,helloworkflow.Starter] was interrupted but is still alive after waiting at least 15000mse

## --- EXAMPLE 2 ---


In [70]:
import subprocess
import time

# Cleanup everything first
!pkill -9 temporal || true
!pkill -f java || true

print("\n🏗️ Setting up Example 2...")
# Move to root, clone, then move inside
%cd /content
# Only clone if the folder doesn't exist yet
![ -d "samples-java" ] || git clone https://github.com/temporalio/samples-java
%cd samples-java

print("🛰️ Terminal 4: Starting 2nd Temporal Server (UI 8082)...")
# Popen runs the server in the background without blocking the cell
# Note: I removed '--port 7235' so it defaults to 7233. This allows the sample code to connect!
server_process_2 = subprocess.Popen([
    'temporal', 'server', 'start-dev',
    '--ui-port', '8082',
    '--db-filename', 'cluster2.db'
])

print("⏳ Waiting 10 seconds for server to boot...")
time.sleep(10)

print("👷 Terminal 5: Building and Executing HelloAccumulator...")
# We use ! here because we WANT the cell to wait for Gradle to finish and print the result
!./gradlew build > /dev/null
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloAccumulator

^C

🏗️ Setting up Example 2...
/content
/content/samples-java
🛰️ Terminal 4: Starting 2nd Temporal Server (UI 8082)...
⏳ Waiting 10 seconds for server to boot...
👷 Terminal 5: Building and Executing HelloAccumulator...
11:34:30.395 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
11:34:31.009 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloAccumulatorTaskQueue", namespace="default", identity=36117@e6322b0888fd} 
11:34:31.044 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloAccumulatorTaskQueue", namespace="default", identity=36117@e6322b0888fd} 
Worker started for task queue: HelloAccumulatorTaskQueue
11:34:31.803 {HelloAccumulatorWorkflow-yellow } [signal sendGreeting] INFO  i.t.s.h.HelloAccumulator$AccumulatorWorkflowImp

In [71]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloAccumulator

11:36:41.050 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
11:36:41.801 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloAccumulatorTaskQueue", namespace="default", identity=36768@e6322b0888fd} 
11:36:41.833 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloAccumulatorTaskQueue", namespace="default", identity=36768@e6322b0888fd} 
Worker started for task queue: HelloAccumulatorTaskQueue
11:36:42.540 {HelloAccumulatorWorkflow-green } [signal sendGreeting] INFO  i.t.s.h.HelloAccumulator$AccumulatorWorkflowImpl - received greeting-Greeting [greetingText=Bill Robot, bucket=green, greetingKey=key-1] 
11:36:42.541 {HelloAccumulatorWorkflow-blue } [signal sendGreeting] INFO  i.t.s.h.HelloAccumulator$AccumulatorWorkflowImpl - rece

In [72]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloActivity


11:38:51.585 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
11:38:52.854 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloActivityTaskQueue", namespace="default", identity=37418@e6322b0888fd} 
11:38:52.905 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloActivityTaskQueue", namespace="default", identity=37418@e6322b0888fd} 
11:38:53.953 {HelloActivityWorkflow 75f41996-ebd7-3393-929c-cc9d9140cd2c} [Activity Executor taskQueue="HelloActivityTaskQueue", namespace="default": 1] INFO  i.t.s.h.HelloActivity$GreetingActivitiesImpl - Composing greeting... 
Hello World!


In [73]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloActivityRetry


11:38:58.754 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
11:38:59.374 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloActivityWithRetriesTaskQueue", namespace="default", identity=37549@e6322b0888fd} 
11:38:59.405 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloActivityWithRetriesTaskQueue", namespace="default", identity=37549@e6322b0888fd} 
composeGreeting activity is going to fail
11:39:00.113 {HelloActivityWithRetriesWorkflow 59115a8e-6af2-367f-8a01-4f5b1ddeff1e} [Activity Executor taskQueue="HelloActivityWithRetriesTaskQueue", namespace="default": 1] WARN  i.t.i.a.ActivityTaskExecutors$ActivityTaskExecutor - Activity failure. ActivityId=59115a8e-6af2-367f-8a01-4f5b1ddeff1e, activityType=ComposeGreeting, attempt=1 

In [74]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloActivityExclusiveChoice


11:39:11.716 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
11:39:12.239 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloActivityChoiceTaskQueue", namespace="default", identity=37699@e6322b0888fd} 
11:39:12.264 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloActivityChoiceTaskQueue", namespace="default", identity=37699@e6322b0888fd} 
Order results: Ordered 5 Bananas...Ordered 8 Apples...Ordered 1 Cherries...Ordered 4 Oranges...


In [75]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloAsync


11:39:17.477 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
11:39:18.153 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloAsyncActivityTaskQueue", namespace="default", identity=37817@e6322b0888fd} 
11:39:18.212 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloAsyncActivityTaskQueue", namespace="default", identity=37817@e6322b0888fd} 
Hello World!
Bye World!


In [76]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloParallelActivity


11:39:25.766 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
11:39:26.402 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloParallelActivityTaskQueue", namespace="default", identity=37940@e6322b0888fd} 
11:39:26.438 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloParallelActivityTaskQueue", namespace="default", identity=37940@e6322b0888fd} 
Hello John!
Hello Mary!
Hello Michael!
Hello Jennet!


In [77]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloAsyncActivityCompletion


11:39:31.853 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
11:39:32.459 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloAsyncActivityCompletionTaskQueue", namespace="default", identity=38070@e6322b0888fd} 
11:39:32.486 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloAsyncActivityCompletionTaskQueue", namespace="default", identity=38070@e6322b0888fd} 
Hello World!


In [78]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloAsyncLambda


11:39:39.257 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
11:39:40.265 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloAsyncLambdaTaskQueue", namespace="default", identity=38192@e6322b0888fd} 
11:39:40.308 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloAsyncLambdaTaskQueue", namespace="default", identity=38192@e6322b0888fd} 
Hello World!
Hello World!


In [79]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloDetachedCancellationScope


11:39:45.925 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
11:39:46.474 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloDetachedCancellationTaskQueue", namespace="default", identity=38321@e6322b0888fd} 
11:39:46.501 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloDetachedCancellationTaskQueue", namespace="default", identity=38321@e6322b0888fd} 
11:39:49.085 {HelloDetachedCancellationWorkflow 77fefd94-4a53-3503-9d2c-30184e29ac61} [Activity Executor taskQueue="HelloDetachedCancellationTaskQueue", namespace="default": 1] INFO  i.t.i.a.ActivityTaskExecutors$ActivityTaskExecutor - Activity canceled. ActivityId=77fefd94-4a53-3503-9d2c-30184e29ac61, activityType=SayHello, attempt=1 
Goodbye John!


In [80]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloChild


11:39:54.242 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
11:39:55.511 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloChildTaskQueue", namespace="default", identity=38445@e6322b0888fd} 
Hello World!


In [81]:
#!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloCron
# taking too long

In [82]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloDynamic


11:40:02.145 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
11:40:02.732 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloDynamicTaskQueue", namespace="default", identity=38573@e6322b0888fd} 
11:40:02.757 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloDynamicTaskQueue", namespace="default", identity=38573@e6322b0888fd} 
DynamicACT: Hello John from: DynamicWF


In [83]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloEagerWorkflowStart


11:40:07.514 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
11:40:08.127 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloEagerWorkflowStartTaskQueue", namespace="default", identity=38687@e6322b0888fd} 
11:40:08.148 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloEagerWorkflowStartTaskQueue", namespace="default", identity=38687@e6322b0888fd} 
11:40:08.904 {HelloEagerWorkflowStartWorkflow e29e6a51-1b7b-30f2-bd44-91008d82cedc} [Activity Executor taskQueue="HelloEagerWorkflowStartTaskQueue", namespace="default": 1] INFO  i.t.s.h.HelloEagerWorkflowStart$GreetingLocalActivitiesImpl - Composing greeting... 
Hello World!


In [84]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloException


11:40:16.286 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
11:40:16.974 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloExceptionTaskQueue", namespace="default", identity=38810@e6322b0888fd} 
11:40:17.006 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloExceptionTaskQueue", namespace="default", identity=38810@e6322b0888fd} 
11:40:17.954 {1cb8a6e9-21eb-3abe-aa1e-60eac1e10520 0de6051c-e691-3843-b10c-d15ec3886fee} [Activity Executor taskQueue="HelloExceptionTaskQueue", namespace="default": 1] WARN  i.t.i.a.ActivityTaskExecutors$ActivityTaskExecutor - Activity failure. ActivityId=0de6051c-e691-3843-b10c-d15ec3886fee, activityType=ComposeGreeting, attempt=1 
java.io.IOException: Hello World!
	at io.temporal.samples.hello.Hel

In [85]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloLocalActivity


11:40:23.254 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
11:40:23.891 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloLocalActivity", namespace="default", identity=38942@e6322b0888fd} 
11:40:23.935 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloLocalActivity", namespace="default", identity=38942@e6322b0888fd} 
Hello World!


In [86]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloPeriodic


11:40:30.216 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
11:40:31.387 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloPeriodicTaskQueue", namespace="default", identity=39062@e6322b0888fd} 
11:40:31.443 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloPeriodicTaskQueue", namespace="default", identity=39062@e6322b0888fd} 
Started a new GreetingWorkflow.
Observing the workflow execution for 20 seconds.
From HelloPeriodicWorkflow: Hello World! I will sleep for 5225 milliseconds and then I will greet you again.
From HelloPeriodicWorkflow: Hello World! I will sleep for 5765 milliseconds and then I will greet you again.
From HelloPeriodicWorkflow: Hello World! I will sleep for 4739 milliseconds and then I will greet you agai

In [87]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloPolymorphicActivity


11:40:56.724 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
11:40:57.379 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloPolymorphicActivityTaskQueue", namespace="default", identity=39267@e6322b0888fd} 
11:40:57.424 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloPolymorphicActivityTaskQueue", namespace="default", identity=39267@e6322b0888fd} 
Hello World!
Bye World!



In [88]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloQuery


11:41:04.898 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
11:41:05.704 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloQueryTaskQueue", namespace="default", identity=39392@e6322b0888fd} 
Hello World!
Bye World!


In [89]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSaga


11:41:13.034 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
11:41:13.689 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloSagaTaskQueue", namespace="default", identity=39517@e6322b0888fd} 
11:41:13.716 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloSagaTaskQueue", namespace="default", identity=39517@e6322b0888fd} 
ActivityOperationImpl.execute() is called with amount 10
ActivityOperationImpl.execute() is called with amount 20
Other compensation logic in main workflow.
ActivityCompensationImpl.compensate() is called with amount -20
ActivityCompensationImpl.compensate() is called with amount -10


In [90]:
#!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSchedules
# taking too long

In [91]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSignal


11:41:22.152 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
11:41:22.807 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloSignalTaskQueue", namespace="default", identity=39645@e6322b0888fd} 
[Hello World!, Hello Universe!]


In [92]:
# !./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSearchAttributes
# taking too long

In [66]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloTypedSearchAttributes


11:33:00.107 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
Exception in thread "main" io.grpc.StatusRuntimeException: UNAVAILABLE: io exception
	at io.grpc.stub.ClientCalls.toStatusRuntimeException(ClientCalls.java:268)
	at io.grpc.stub.ClientCalls.getUnchecked(ClientCalls.java:249)
	at io.grpc.stub.ClientCalls.blockingUnaryCall(ClientCalls.java:167)
	at io.temporal.api.workflowservice.v1.WorkflowServiceGrpc$WorkflowServiceBlockingStub.getSystemInfo(WorkflowServiceGrpc.java:6802)
	at io.temporal.serviceclient.SystemInfoInterceptor.getServerCapabilitiesOrThrow(SystemInfoInterceptor.java:112)
	at io.temporal.serviceclient.SystemInfoInterceptor$1.start(SystemInfoInterceptor.java:66)
	at io.grpc.ForwardingClientCall.start(ForwardingClientCall.java:32)
	at io.temporal.serviceclient.GrpcMetadataProviderInterceptor$HeaderAttachingClientCall.start(GrpcMetadata

In [93]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSideEffect



11:41:27.791 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
11:41:28.429 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloSideEffectTaskQueue", namespace="default", identity=39761@e6322b0888fd} 
11:41:28.460 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloSideEffectTaskQueue", namespace="default", identity=39761@e6322b0888fd} 
Goodbye World!
Goodbye World!


In [94]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloUpdate


11:41:33.940 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
11:41:34.997 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloUpdateTaskQueue", namespace="default", identity=39882@e6322b0888fd} 
11:41:35.039 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloUpdateTaskQueue", namespace="default", identity=39882@e6322b0888fd} 
11:41:36.416 {HelloUpdateWorkflow b5817c2c-dc38-3bec-8299-4a3f5fbda18f} [Activity Executor taskQueue="HelloUpdateTaskQueue", namespace="default": 1] INFO  i.t.s.h.HelloActivity$GreetingActivitiesImpl - Composing greeting... 
11:41:36.651 {HelloUpdateWorkflow 9473750f-f16c-350c-a8f8-46ef7e7902ba} [Activity Executor taskQueue="HelloUpdateTaskQueue", namespace="default": 1] INFO  i.t.s.h.HelloActivity$Greetin

In [95]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSignalWithTimer


11:41:42.197 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
11:41:42.710 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloSignalWithTimerTaskQueue", namespace="default", identity=40009@e6322b0888fd} 
11:41:42.738 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloSignalWithTimerTaskQueue", namespace="default", identity=40009@e6322b0888fd} 
Processing value: Value 2
Processing value: Value 5
Processing value: Value 7
Processing value: Value 10
11:42:08.417 {HelloSignalWithTimerWorkflow } [workflow-method-HelloSignalWithTimerWorkflow-e3349f07-018e-4d64-b0e9-733dfc040e4b] INFO  i.t.s.h.HelloSignalWithTimer$SignalWithTimerWorkflowImpl - Timer canceled via exit signal 
Processing value: Value 11


In [96]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSignalWithStartAndWorkflowInit

11:42:12.734 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
11:42:13.366 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloWithInitTaskQueue", namespace="default", identity=40231@e6322b0888fd} 
11:42:13.398 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloWithInitTaskQueue", namespace="default", identity=40231@e6322b0888fd} 
Result: Hello Michael Jordan,Hello John Stockton
11:42:14.495 {without-init } [signal addGreeting] WARN  i.t.i.sync.WorkflowExecutionHandler - Workflow execution failure WorkflowId='without-init', RunId=21874d4d-bd28-4748-8691-0ad575a649d3, WorkflowType='MyWorkflowNoInit' 
java.lang.NullPointerException: Cannot invoke "java.util.List.add(Object)" because "this.peopleToGreet" is null
	at io.temporal.sam

In [97]:
# Cleanup
!pkill -9 temporal || true
!pkill -f java || true

^C
